# Debug Inference Viewer

Load a debug subfolder from `debug_output/` and overlay interaction
channels on the volume. Uses smart slice selection: ranks all
(axis, slice) pairs by in-plane scribble density so each scribble
is shown in its drawn plane.

In [ ]:
import json
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

## Configuration

Set `debug_dir` to the timestamped subfolder you want to inspect.
List available subfolders below.

In [ ]:
debug_root = Path('debug_output')
available = sorted(debug_root.iterdir()) if debug_root.exists() else []
for d in available:
    print(d.name)

In [ ]:
# Pick the most recent subfolder (or set manually)
debug_dir = available[-1] if available else None
assert debug_dir is not None, "No debug subfolders found in debug_output/"
print(f"Loading: {debug_dir}")

In [ ]:
with open(debug_dir / 'metadata.json') as f:
    metadata = json.load(f)
print(json.dumps(metadata, indent=2))

## Load channels

In [ ]:
channels = {}
for nii_path in sorted(debug_dir.glob('*.nii.gz')):
    name = nii_path.name.replace('.nii.gz', '')
    channels[name] = nib.load(str(nii_path)).get_fdata().astype(np.float32)
    print(f"{name}: shape={channels[name].shape}, "
          f"range=[{channels[name].min():.4f}, {channels[name].max():.4f}]")

## Smart slice selection

Rank all (axis, slice) pairs by in-plane scribble density, filtering
out cross-axis views where scribbles appear as thin cross-sections.

In [ ]:
AXIS_NAMES = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}
MIN_THICKNESS = 3
MAX_SLICES = 8

volume = channels['0_volume']
positive = channels.get('2_positive', np.zeros_like(volume))
negative = channels.get('3_negative', np.zeros_like(volume))
combined = positive + negative

candidates = []
for axis in range(3):
    for s in range(combined.shape[axis]):
        mask = np.take(combined, s, axis=axis)
        if not mask.any():
            continue
        density = mask.sum()
        binary = mask > 0
        rows = binary.any(axis=1).sum()
        cols = binary.any(axis=0).sum()
        if min(rows, cols) >= MIN_THICKNESS:
            candidates.append((axis, s, density))

candidates.sort(key=lambda x: x[2], reverse=True)
selected = [(a, s) for a, s, _ in candidates[:MAX_SLICES]]
selected.sort(key=lambda x: (x[0], x[1]))

# Fallback: if no in-plane scribbles, just take top density slices
if not selected:
    fallback = []
    for axis in range(3):
        for s in range(combined.shape[axis]):
            mask = np.take(combined, s, axis=axis)
            d = mask.sum()
            if d > 0:
                fallback.append((axis, s, d))
    fallback.sort(key=lambda x: x[2], reverse=True)
    selected = [(a, s) for a, s, _ in fallback[:MAX_SLICES]]
    selected.sort(key=lambda x: (x[0], x[1]))
    print("WARNING: no in-plane scribbles found, using fallback")

print(f"Selected {len(selected)} slices:")
for axis, s in selected:
    mask = np.take(combined, s, axis=axis)
    print(f"  {AXIS_NAMES[axis]} #{s}: density={mask.sum():.0f}")

## Overlay interactions on volume

Green = positive interactions, Red = negative interactions.
Slices selected by in-plane scribble density across all axes.

In [ ]:
pos_cmap = mcolors.LinearSegmentedColormap.from_list(
    'vibrant_green', [(0, 0, 0, 0), (0.0, 0.9, 0.0, 1.0)],
)
neg_cmap = mcolors.LinearSegmentedColormap.from_list(
    'vibrant_red', [(0, 0, 0, 0), (0.9, 0.0, 0.0, 1.0)],
)

n_cols = len(selected)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
if n_cols == 1:
    axes = [axes]

for ax_idx, (axis, sl) in enumerate(selected):
    ax = axes[ax_idx]
    vol_slice = np.take(volume, sl, axis=axis)
    pos_slice = np.take(positive, sl, axis=axis)
    neg_slice = np.take(negative, sl, axis=axis)

    ax.imshow(vol_slice.T, cmap='gray', origin='lower')

    if pos_slice.max() > 0:
        ax.imshow(
            np.ma.masked_where(pos_slice.T < 0.01, pos_slice.T),
            cmap=pos_cmap, alpha=0.9, origin='lower', vmin=0, vmax=1,
        )

    if neg_slice.max() > 0:
        ax.imshow(
            np.ma.masked_where(neg_slice.T < 0.01, neg_slice.T),
            cmap=neg_cmap, alpha=0.9, origin='lower', vmin=0, vmax=1,
        )

    n_pos = int(pos_slice.sum())
    n_neg = int(neg_slice.sum())
    ax.set_title(f'{AXIS_NAMES[axis]} #{sl}\n+{n_pos} / -{n_neg}', fontsize=10)
    ax.axis('off')

fig.suptitle(
    f"Model: {metadata['model_name']}  |  {metadata['timestamp']}\n"
    f"Green = positive, Red = negative  |  "
    f"{int(positive.sum())} pos, {int(negative.sum())} neg voxels total",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(debug_dir / 'overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## Previous prediction channel

In [ ]:
prev_pred = channels.get('1_previous_pred')
if prev_pred is not None and prev_pred.max() > 0:
    mid_z = prev_pred.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, offset in zip(axes, [-5, 0, 5]):
        sl = min(max(mid_z + offset, 0), prev_pred.shape[2] - 1)
        ax.imshow(prev_pred[:, :, sl].T, cmap='hot', origin='lower')
        ax.set_title(f'prev_pred z={sl}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No previous prediction data (channel is all zeros).")